# Ising vs Heisenberg vs XY — Three-Way Comparison

Systematic comparison of three spin models on BCC Fe and FCC Ni crystal graphs.

## Models
| Model | Spin symmetry | Algorithm | Universality |
|-------|--------------|-----------|---------------|
| Ising | Z(2) discrete | Wolff cluster | Z(2) 3D |
| XY | O(2) planar | Wolff cluster | O(2) 3D |
| Heisenberg | O(3) vector | Overrelaxation+Metropolis | O(3) 3D |

## Sections
1. Tc ratios: BCC and FCC across all three models
2. Critical exponent comparison (theory values)
3. J_fit comparison vs Pajda 2001 ab initio
4. Physical interpretation

In [ ]:
## Section 1 — Tc comparison bar charts

import numpy as np
import matplotlib.pyplot as plt

KB_EV    = 8.617333e-5   # eV/K
KB_MEV   = KB_EV * 1000  # meV/K
TC_FE    = 1043.0        # K  experimental
TC_NI    = 627.0         # K  experimental
J_LIT_FE = 16.3          # meV  Pajda 2001, BCC Fe first-NN
J_LIT_NI = 4.1           # meV  Pajda 2001, FCC Ni first-NN

# Tc values in J/kB units (from FSS Binder crossings)
Tc_vals = {
    'Ising':       {'BCC': (6.3548, 0.1250), 'FCC': (9.8924, 0.2010)},
    'XY':          {'BCC': (None, None),      'FCC': (None, None)},      # populate from xy_jfit.ipynb
    'Heisenberg':  {'BCC': (None, None),      'FCC': (None, None)},      # populate from heisenberg_jfit.ipynb
}

models  = ['Ising', 'XY', 'Heisenberg']
colors  = ['steelblue', 'darkorange', 'seagreen']
x       = np.arange(len(models))
width   = 0.55

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, lattice in zip(axes, ['BCC', 'FCC']):
    vals = []
    errs = []
    for m in models:
        tc, err = Tc_vals[m][lattice]
        vals.append(tc if tc is not None else 0.0)
        errs.append(err if err is not None else 0.0)

    bars = ax.bar(x, vals, width, yerr=errs, capsize=6, color=colors,
                  alpha=0.85, edgecolor='k', linewidth=0.8)

    for bar, m, val, err in zip(bars, models, vals, errs):
        tc_raw, _ = Tc_vals[m][lattice]
        if tc_raw is None:
            ax.text(bar.get_x() + bar.get_width()/2, 0.05,
                    'TBD', ha='center', va='bottom', fontsize=9, color='gray', fontstyle='italic')
        else:
            label_str = f'{val:.3f}\n±{err:.4f}'
            ax.text(bar.get_x() + bar.get_width()/2, val + max(errs)*0.05 + 0.05,
                    label_str, ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=10)
    ax.set_ylabel('Tc  (J/kB)')
    ax.set_title(f'{lattice} lattice — Tc(J=1) by model', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Tc comparison: Ising vs XY vs Heisenberg', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Ratio plot: Tc_Ising / Tc_other ---
fig2, axes2 = plt.subplots(1, 2, figsize=(10, 4))

for ax, lattice in zip(axes2, ['BCC', 'FCC']):
    tc_ising, _ = Tc_vals['Ising'][lattice]
    ratios = []
    ratio_errs = []
    for m in models:
        tc, err = Tc_vals[m][lattice]
        if tc is not None and tc_ising is not None:
            r = tc_ising / tc
            tc_i_err = Tc_vals['Ising'][lattice][1]
            r_err = r * np.sqrt((tc_i_err/tc_ising)**2 + (err/tc)**2)
        else:
            r, r_err = None, None
        ratios.append(r if r is not None else 0.0)
        ratio_errs.append(r_err if r_err is not None else 0.0)

    bars = ax.bar(x, ratios, width, yerr=ratio_errs, capsize=6, color=colors,
                  alpha=0.85, edgecolor='k', linewidth=0.8)

    for bar, m, r in zip(bars, models, ratios):
        raw_tc, _ = Tc_vals[m][lattice]
        if raw_tc is None:
            ax.text(bar.get_x() + bar.get_width()/2, 0.05,
                    'TBD', ha='center', va='bottom', fontsize=9, color='gray', fontstyle='italic')
        else:
            ax.text(bar.get_x() + bar.get_width()/2, r + 0.03,
                    f'{r:.3f}', ha='center', va='bottom', fontsize=9)

    ax.axhline(1.0, color='k', linestyle='--', alpha=0.4, linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=10)
    ax.set_ylabel('Tc_Ising / Tc_model')
    ax.set_title(f'{lattice} — Tc ratio (Ising = reference)', fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Tc ratio: Ising as reference (ratio=1 dashed)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print('Note: XY and Heisenberg Tc values are placeholders (None → 0-height bars).')
print('Run xy_jfit.ipynb and heisenberg_jfit.ipynb to populate Tc_vals.')

In [ ]:
## Section 2 — Critical exponent comparison

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Theory FSS exponents for 3D universality classes
# Sources: Hasenbusch 2001 (Ising), Campostrini 2002 (XY/Heisenberg)
exponents = {
    #  model     symmetry   Tc(cubic)  nu       gamma    beta
    'Ising':      {'symmetry': 'Z(2)', 'Tc': 4.5116, 'nu': 0.6301, 'gamma': 1.2372, 'beta': 0.3265},
    'XY':         {'symmetry': 'O(2)', 'Tc': 2.2016, 'nu': 0.6717, 'gamma': 1.3177, 'beta': 0.3486},
    'Heisenberg': {'symmetry': 'O(3)', 'Tc': 1.4432, 'nu': 0.7112, 'gamma': 1.3960, 'beta': 0.3646},
}

# Print formatted table
print('=' * 72)
print(f'{"Model":<14} {"Symmetry":<10} {"Tc(cubic)":>10} {"nu":>8} {"gamma":>8} {"beta":>8}')
print('-' * 72)
for model, d in exponents.items():
    print(f'{model:<14} {d["symmetry"]:<10} {d["Tc"]:>10.4f} {d["nu"]:>8.4f} {d["gamma"]:>8.4f} {d["beta"]:>8.4f}')
print('=' * 72)
print('Sources: Hasenbusch (2001), Campostrini et al. (2002)')
print('Tc(cubic) is the critical temperature of the simple-cubic model at J=1.')

# Grouped bar chart: nu, gamma, beta for three models
exp_keys   = ['nu', 'gamma', 'beta']
exp_labels = [r'$\nu$', r'$\gamma$', r'$\beta$']
model_list = ['Ising', 'XY', 'Heisenberg']
bar_colors = ['steelblue', 'darkorange', 'seagreen']

x     = np.arange(len(exp_keys))
width = 0.22
offsets = [-width, 0, width]

fig, ax = plt.subplots(figsize=(9, 5))

for i, (model, color, offset) in enumerate(zip(model_list, bar_colors, offsets)):
    vals = [exponents[model][k] for k in exp_keys]
    bars = ax.bar(x + offset, vals, width, label=model, color=color,
                  alpha=0.85, edgecolor='k', linewidth=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{val:.4f}', ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(exp_labels, fontsize=13)
ax.set_ylabel('Exponent value')
ax.set_title('Critical exponents: Ising Z(2) vs XY O(2) vs Heisenberg O(3)', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.65)

plt.tight_layout()
Path('figures').mkdir(exist_ok=True)
plt.savefig('figures/three_way_exponents.png', dpi=150)
plt.show()
print('Saved figures/three_way_exponents.png')

In [ ]:
## Section 3 — J_fit comparison vs Pajda 2001

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

KB_EV    = 8.617333e-5
KB_MEV   = KB_EV * 1000
TC_FE    = 1043.0
TC_NI    = 627.0
J_LIT_FE = 16.3
J_LIT_NI = 4.1

# Ising J_fit (BCC=Fe, FCC=Ni)
J_ising_Fe     = KB_MEV * TC_FE / 6.3548
J_ising_Fe_err = J_ising_Fe * (0.1250 / 6.3548)
J_ising_Ni     = KB_MEV * TC_NI / 9.8924
J_ising_Ni_err = J_ising_Ni * (0.2010 / 9.8924)

# Run xy_jfit.ipynb and heisenberg_jfit.ipynb to populate:
J_xy_Fe,   J_xy_Fe_err   = None, None
J_xy_Ni,   J_xy_Ni_err   = None, None
J_heis_Fe, J_heis_Fe_err = None, None
J_heis_Ni, J_heis_Ni_err = None, None

def pct_err(val, ref):
    return 100 * abs(val - ref) / ref if val is not None else float('nan')

print('=' * 80)
print(f'{"Material":<12} {"Model":<14} {"Tc(J=1)":>10} {"J_fit (meV)":>16} {"J_Pajda":>9} {"Error":>8}')
print('-' * 80)

rows = [
    ('Fe (BCC)', 'Ising',       6.3548, J_ising_Fe,  J_ising_Fe_err,  J_LIT_FE),
    ('Fe (BCC)', 'XY',          None,   J_xy_Fe,     J_xy_Fe_err,     J_LIT_FE),
    ('Fe (BCC)', 'Heisenberg',  None,   J_heis_Fe,   J_heis_Fe_err,   J_LIT_FE),
    ('Ni (FCC)', 'Ising',       9.8924, J_ising_Ni,  J_ising_Ni_err,  J_LIT_NI),
    ('Ni (FCC)', 'XY',          None,   J_xy_Ni,     J_xy_Ni_err,     J_LIT_NI),
    ('Ni (FCC)', 'Heisenberg',  None,   J_heis_Ni,   J_heis_Ni_err,   J_LIT_NI),
]

for mat, model, tc, j, jerr, jlit in rows:
    tc_str  = f'{tc:.4f}'   if tc   is not None else 'TBD'
    j_str   = f'{j:.2f}±{jerr:.2f}' if j is not None else 'TBD'
    err_str = f'{pct_err(j, jlit):.1f}%' if j is not None else 'TBD'
    print(f'{mat:<12} {model:<14} {tc_str:>10} {j_str:>16} {jlit:>9.1f} {err_str:>8}')

print('=' * 80)
print('Literature: Pajda et al., Phys. Rev. B 64, 174402 (2001)')
print('Note: XY and Heisenberg Tc values are TBD — run xy_jfit.ipynb / heisenberg_jfit.ipynb.')

# Bar chart — Fe and Ni subplots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

bar_labels = ['Pajda LSWT\n(quantum)', 'Ising\n(classical)', 'XY\n(classical)', 'Heisenberg\n(classical)']
bar_colors = ['steelblue', 'darkorange', 'orchid', 'seagreen']

for ax, (mat, j_ising, j_ising_err, j_xy, j_xy_err, j_heis, j_heis_err, j_lit) in zip(axes, [
    ('Fe (BCC)', J_ising_Fe, J_ising_Fe_err, J_xy_Fe, J_xy_Fe_err, J_heis_Fe, J_heis_Fe_err, J_LIT_FE),
    ('Ni (FCC)', J_ising_Ni, J_ising_Ni_err, J_xy_Ni, J_xy_Ni_err, J_heis_Ni, J_heis_Ni_err, J_LIT_NI),
]):
    vals_raw = [j_lit,     j_ising,     j_xy,     j_heis]
    errs_raw = [0,         j_ising_err, j_xy_err, j_heis_err]

    vals = [v if v is not None else 0.0 for v in vals_raw]
    errs = [e if e is not None else 0.0 for e in errs_raw]

    bars = ax.bar(bar_labels, vals, yerr=errs, capsize=6, color=bar_colors,
                  alpha=0.85, edgecolor='k', linewidth=0.8)

    max_val = max(v for v in vals if v > 0)
    for bar, val, err, val_raw in zip(bars, vals, errs, vals_raw):
        if val_raw is None:
            ax.text(bar.get_x() + bar.get_width()/2, 0.1,
                    'TBD', ha='center', va='bottom', fontsize=9, color='gray', fontstyle='italic')
        else:
            label_str = f'{val:.1f}' if err == 0 else f'{val:.1f}±{err:.1f}'
            ax.text(bar.get_x() + bar.get_width()/2, val + max_val * 0.02 + 0.1,
                    label_str, ha='center', va='bottom', fontsize=9)

    ax.set_ylabel('J  (meV)')
    ax.set_title(mat, fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, max_val * 1.35)

plt.suptitle('J_fit comparison: Ising vs XY vs Heisenberg vs Pajda 2001 LSWT', fontsize=12, fontweight='bold')
plt.tight_layout()
Path('figures').mkdir(exist_ok=True)
plt.savefig('figures/three_way_jfit.png', dpi=150)
plt.show()
print('Saved figures/three_way_jfit.png')

In [ ]:
## Section 4 — Physical interpretation

print("""
Physical interpretation
=======================

Spin symmetry and Tc (J=1):
  Ising Z(2) > XY O(2) > Heisenberg O(3)

  The ordering Tc(Ising) > Tc(XY) > Tc(Heisenberg) for fixed J and lattice
  reflects the role of spin dimensionality in suppressing order.

  - Ising Z(2): spins are discrete scalars (up/down). No continuous fluctuations
    are possible transverse to the order parameter. Thermal fluctuations can only
    invert spins entirely, which costs O(zJ) energy. This makes ordering robust
    and pushes Tc to the highest value per unit J.

  - XY O(2): spins are 2D planar vectors. One transverse (Goldstone-like) mode
    exists, allowing low-cost spin-wave fluctuations that partially disorder the
    system. Tc/J is intermediate between Ising and Heisenberg.

  - Heisenberg O(3): spins are full 3D vectors. Two transverse fluctuation
    modes exist, providing the most efficient channel for destroying long-range
    order. This maximally suppresses Tc/J, yielding the lowest critical
    temperature per unit exchange coupling.

Implications for J_fit:
  J_fit = kB * Tc_exp / Tc_sim(J=1)

  Because Tc_sim(Ising) >> Tc_sim(Heisenberg) >> Tc_sim(XY) (for the same J),
  the inferred J_fit from Ising is the SMALLEST and from Heisenberg the LARGEST.
  XY lies between them.

  In other words:
    J_fit(Ising) < J_fit(XY) < J_fit(Heisenberg)

  The Ising model needs a smaller J to reach the same experimental Tc because
  its discrete symmetry already locks spins more tightly. The Heisenberg model
  needs a larger J to compensate for the extra thermal disorder from 3 spin
  components.

Mermin-Wagner theorem (conceptual link):
  In 2D, the Mermin-Wagner theorem forbids spontaneous breaking of continuous
  symmetries (XY or Heisenberg) at any finite T. Ising, having discrete Z(2)
  symmetry, is exempt and has a finite Tc in 2D. In 3D all three models order,
  but the theorem's logic carries over qualitatively: continuous symmetry
  (XY, Heisenberg) is harder to break, reducing Tc/J relative to discrete
  symmetry (Ising).

Which model is most appropriate for Fe and Ni?
  Physically: Heisenberg.
    Fe (3d^6) and Ni (3d^8) are itinerant ferromagnets with nearly-quenched
    orbital moments. Their spin operators transform as O(3) vectors under
    rotation, so the Heisenberg model captures the correct spin symmetry.
    The Heisenberg J_fit is the most physically motivated comparison to
    Pajda 2001 LSWT (which also uses vector spins in a Heisenberg Hamiltonian).

  Empirically: Ising is commonly used in first-principles Tc calculations.
    Some DFT workflows (e.g., MFT-based codes) default to a scalar order
    parameter and quote Ising Tc. This is expedient but introduces a model
    error of roughly Tc_Ising/Tc_Heisenberg ~ 3–4× in the effective Tc/J ratio.

  XY is relevant if spin-orbit coupling constrains spins to a plane (easy-plane
  anisotropy). Neither Fe nor Ni is strongly easy-plane, so XY is not the
  primary model choice, though it brackets Heisenberg usefully.

Summary table of model hierarchy:
  Property             Ising Z(2)   XY O(2)    Heisenberg O(3)
  ---------------------------------------------------------------
  Tc/J (BCC approx)    6.35         ~2.8        ~2.1
  J_fit (Fe/BCC)       ~14 meV      ~TBD        ~TBD
  Physical realism     Low          Medium      High
  Computational cost   Low          Medium      High
  Closest to Pajda     Coincidental  Intermediate  Directly comparable
""")